# Can we predict arrears next month?
Target: `arrears_next_month`. Methodology: `checklist.md`.

**Feature decisions (checklist 7).** `meter_ref` is an identifier (used only for grouping, never as a feature). `billing_month` is used only for splitting. `final_notice_sent` is recorded in the billing cycle *after* the billing month, so it is not known at prediction time, and it is nearly a proxy for the target; it is dropped (leakage). Features: `tariff`, `occupants`, `kwh`, `bill_eur`, `late_payments_12m`.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.base import clone
df = pd.read_csv('data.csv', parse_dates=['billing_month'])
print(df.shape, df.meter_ref.nunique(), df.billing_month.min().date(), df.billing_month.max().date())
print('arrears rate', df.arrears_next_month.mean().round(3))
print('rate by final_notice_sent (leaky, not used):')
print(df.groupby('final_notice_sent').arrears_next_month.mean())

## 1. Test set, set aside before any modelling
The model will be used on later months, and accounts repeat across many rows. So the test set is **both** later in time and whole accounts: a random 20% of accounts are test accounts, and test rows are their rows in the last 4 billing months (Sep-Dec 2025). Training rows are rows of the *other* accounts that are before that period. Test accounts never appear in training.

In [ ]:
rng = np.random.RandomState(0)
ids = df.meter_ref.unique()
test_ids = set(rng.choice(ids, size=int(.2 * len(ids)), replace=False))
cut = pd.Timestamp('2025-09-01')
is_test = df.meter_ref.isin(test_ids) & (df.billing_month >= cut)
is_train = ~df.meter_ref.isin(test_ids) & (df.billing_month < cut)
test, train = df[is_test].reset_index(drop=True), df[is_train].reset_index(drop=True)
assert not set(test.meter_ref) & set(train.meter_ref) and train.billing_month.max() < test.billing_month.min()
print(len(train), len(test), train.arrears_next_month.mean().round(3), test.arrears_next_month.mean().round(3))

## 2. Cross-validation inside the training rows only
Same structure as the test split. Training accounts are randomly divided into 5 groups. Fold k validates on group k's rows in the k-th of the last five 3-month blocks, and trains only on the *other* groups' rows from months before that block. So earlier rows train and later rows validate, and validation accounts are never in the training part of the fold.

In [ ]:
months = np.sort(train.billing_month.unique())
blocks = [months[i:i + 3] for i in range(len(months) - 15, len(months), 3)]
tids = train.meter_ref.unique(); rng2 = np.random.RandomState(1); rng2.shuffle(tids)
grp = {m: i % 5 for i, m in enumerate(tids)}
train_grp = train.meter_ref.map(grp).values
def folds():
    for k, b in enumerate(blocks):
        va = train.billing_month.isin(b).values & (train_grp == k)
        tr = (train.billing_month < b[0]).values & (train_grp != k)
        yield np.where(tr)[0], np.where(va)[0]
for tr, va in folds():
    print(len(tr), len(va))

## 3-4. Candidate models (all named before any is scored)
1. `baseline`: predicts the training prior (trivial).
2. `logreg`: standardised numerics + one-hot tariff, logistic regression (C=1).
3. `logreg_C0.1`: same with stronger regularisation.
4. `hgb`: gradient boosting, max_depth 3, lr 0.05, 200 iterations.
5. `hgb_deep`: gradient boosting, max_depth 6, min_samples_leaf 100.

Primary metric: average precision (arrears are ~12% of rows); ROC-AUC also reported. The decision threshold is chosen on pooled out-of-fold predictions to maximise F1. Preprocessing sits inside each pipeline, so it is fit within each fold (checklist 3).

In [ ]:
num = ['occupants', 'kwh', 'bill_eur', 'late_payments_12m']; cat = ['tariff']; FEATS = num + cat
def lin(C):
    return Pipeline([('p', ColumnTransformer([('n', StandardScaler(), num), ('c', OneHotEncoder(handle_unknown='ignore'), cat)])), ('m', LogisticRegression(C=C, max_iter=1000))])
def gb(**k):
    return Pipeline([('p', ColumnTransformer([('n', 'passthrough', num), ('c', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat)])), ('m', HistGradientBoostingClassifier(learning_rate=.05, max_iter=200, random_state=0, **k))])
models = {'baseline': Pipeline([('m', DummyClassifier(strategy='prior'))]),
          'logreg': lin(1), 'logreg_C0.1': lin(.1),
          'hgb': gb(max_depth=3), 'hgb_deep': gb(max_depth=6, min_samples_leaf=100)}
X, y = train[FEATS], train.arrears_next_month.values
oof = {}; res = []
for n, m in models.items():
    p = np.full(len(train), np.nan); ap = []; auc = []
    for tr, va in folds():
        mm = clone(m).fit(X.iloc[tr], y[tr]); p[va] = mm.predict_proba(X.iloc[va])[:, 1]
        ap.append(average_precision_score(y[va], p[va])); auc.append(roc_auc_score(y[va], p[va]))
    oof[n] = p; res.append((n, np.mean(ap), np.std(ap), np.mean(auc)))
cv = pd.DataFrame(res, columns=['model', 'cv_AP', 'AP_sd', 'cv_AUC']).set_index('model')
cv.round(3)

## 5. Choice, on cross-validation only

In [ ]:
best = cv.cv_AP.idxmax(); mask = ~np.isnan(oof[best])
ths = np.linspace(.02, .8, 79)
f1s = [f1_score(y[mask], oof[best][mask] > t) for t in ths]
THR = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(THR, 2), '| CV F1:', round(max(f1s), 3))

**Decision (written before the test set is scored):** the model with the highest mean CV average precision, with the features listed above and the F1-maximising out-of-fold threshold printed above. No tuning follows. If the best model does not clearly beat the baseline in CV, the honest conclusion is that arrears are not predictable from these features.

## 6. Refit on all training rows, score the test set once

In [ ]:
final = clone(models[best]).fit(X, y)
pt = final.predict_proba(test[FEATS])[:, 1]; yt = test.arrears_next_month.values
prior = clone(models['baseline']).fit(X, y).predict_proba(test[FEATS])[:, 1]
print(f'TEST  AP={average_precision_score(yt, pt):.3f} (baseline {average_precision_score(yt, prior):.3f})  AUC={roc_auc_score(yt, pt):.3f}')
pr = pt >= THR
print(f'at threshold {THR:.2f}: precision={precision_score(yt, pr):.3f} recall={recall_score(yt, pr):.3f} F1={f1_score(yt, pr):.3f}')

## Conclusion
The test line above is the single headline (scored once, nothing changed afterwards). Compare its AP with the baseline AP (the test prevalence) to judge lift. Caveats: `final_notice_sent` was excluded as post-billing information; including it would score much higher but could not be used at prediction time. The test set is 4 months of a random fifth of accounts, so the estimate carries sampling noise.